<a href="https://colab.research.google.com/github/kartikigaikwad/Amazon-Clone/blob/main/Capstone_task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Cell 1: Install libs (runs in Colab)
!pip install --quiet langgraph graphviz arxiv wikipedia openai tiktoken
# graphviz system package for rendering (Colab usually has it)
!apt-get -qq update && apt-get -qq install -y graphviz

# NOTE: langgraph API may evolve; code below will attempt to import and fallback gracefully.
print("Installed base packages. Restart runtime if any import fails.")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Installed base packages. Restart runtime if any import fails.


In [3]:
# Cell 2: imports and utilities
import time, json, textwrap, os, math, uuid
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any, Optional, Tuple
import logging
logging.basicConfig(level=logging.INFO)

# Optional libs
try:
    import langgraph
    HAS_LANGGRAPH = True
    logging.info("langgraph available.")
except Exception as e:
    langgraph = None
    HAS_LANGGRAPH = False
    logging.info("langgraph NOT available — falling back to local orchestrator.")

import arxiv
import wikipedia
from graphviz import Digraph

# --- Token / budget helpers (rough approximation) ---
def estimate_tokens(text: str) -> int:
    # rough estimate: 1 token ~= 4 chars
    return max(1, len(text) // 4)

@dataclass
class TokenBudget:
    total_budget: int = 100000  # global budget per project
    used: int = 0
    def charge(self, amount:int):
        self.used += amount
    def remaining(self) -> int:
        return self.total_budget - self.used


In [4]:
# Cell 3: state schema
@dataclass
class Task:
    id: str
    title: str
    query: str
    section: str
    status: str = "pending"
    results: List[Dict[str,Any]] = field(default_factory=list)
    summary: Optional[str] = None

@dataclass
class ProjectState:
    project_id: str
    prompt: str
    plan: List[Task] = field(default_factory=list)
    iterations: int = 0
    max_iterations: int = 5
    logs: List[str] = field(default_factory=list)
    final_draft: Optional[str] = None
    bibliography: List[Dict[str,Any]] = field(default_factory=list)
    token_budget: TokenBudget = field(default_factory=lambda: TokenBudget(total_budget=50000))

    def log(self, msg: str):
        t = time.strftime("%Y-%m-%d %H:%M:%S")
        self.logs.append(f"[{t}] {msg}")
        logging.info(msg)


In [5]:
# Cell 4: Planner Agent
import uuid
def planner_agent(state: ProjectState) -> Tuple[ProjectState, Dict]:
    """
    Decompose the user prompt into a small list of tasks (search queries + sections).
    Returns updated state and plan metadata for HIL display.
    """
    state.log("Planner: starting task decomposition.")
    prompt = state.prompt.lower()
    tasks = []

    # Very simple heuristics to create tasks — replace with LLM prompt call for production
    # Task templates: historical, definitions, comparisons, methods, recent developments, gaps
    templates = [
        ("History / background", f"history of {prompt}"),
        ("Definitions & scope", f"what is {prompt} and scope"),
        ("Key methods / approaches", f"methods used in {prompt}"),
        ("Compare major approaches", f"compare different approaches used in {prompt}"),
        ("Recent advances (last 5 years)", f"recent advances in {prompt} 2020..2025"),
        ("Open challenges & future directions", f"open problems in {prompt}")
    ]
    # prune templates if prompt small
    chosen = templates[:5] if len(prompt.split())>1 else templates[:3]

    for title, q in chosen:
        tasks.append(Task(id=str(uuid.uuid4()), title=title, query=q, section=title))

    state.plan = tasks
    state.log(f"Planner: generated {len(tasks)} tasks.")
    # pack a human-readable plan
    plan_view = [{"id": t.id, "title": t.title, "query": t.query, "section": t.section} for t in tasks]
    return state, {"plan": plan_view}


In [6]:
# Cell 5: HIL Node 1 (Plan approval/reject)
def hil1_review(plan_meta: Dict) -> Tuple[bool, Optional[str]]:
    """
    Present plan to human, accept or reject with feedback.
    For Colab simple input() is used.
    Returns: (accepted:bool, feedback:str|None)
    """
    print("\n=== PLAN PREVIEW ===")
    for i, t in enumerate(plan_meta["plan"],1):
        print(f"{i}. {t['title']}  — query: {t['query']}")
    print("====================")
    resp = input("HIL1: Accept plan? (y/n): ").strip().lower()
    if resp == 'y':
        return True, None
    else:
        fb = input("Please provide feedback for replanning (short): ").strip()
        return False, fb


In [7]:
# Cell 6: Researcher Agent
# NOTE: This cell supports two modes:
#  - mock mode (no API keys): uses wikipedia/arXiv simple searches and summaries
#  - live mode: you can plug in SerpAPI/Tavily for web searches and more sources.

def researcher_agent(task: Task, state: ProjectState, mode="auto") -> Task:
    """
    Execute search for a Task. Populates task.results and task.summary.
    mode: "auto" uses arXiv + wikipedia; "mock" uses wikipedia; "serp" can use SerpAPI (placeholder).
    """
    state.log(f"Researcher: starting task '{task.title}' (query='{task.query}').")
    results = []
    # 1) Wikipedia search (fast)
    try:
        wq = task.query
        wiki_title = None
        try:
            wiki_title = wikipedia.search(wq, results=3)
        except Exception:
            wiki_title = []
        for wt in (wiki_title or [])[:2]:
            try:
                txt = wikipedia.summary(wt, sentences=3)
                results.append({"source":"wikipedia", "title":wt, "snippet":txt, "url":f"https://en.wikipedia.org/wiki/{wt.replace(' ','_')}"})
            except Exception:
                pass
    except Exception as e:
        state.log(f"Researcher: wikipedia error: {e}")

    # 2) arXiv search for academic content
    try:
        # arxiv library search (limit results)
        search = arxiv.Search(query=task.query, max_results=3, sort_by=arxiv.SortCriterion.Relevance)
        for r in search.results():
            summary = r.summary[:800]
            results.append({"source":"arXiv", "title":r.title, "snippet": summary, "url": r.entry_id, "authors":[a.name for a in r.authors], "year": getattr(r,'published',None).year if getattr(r,'published',None) else None})
    except Exception as e:
        state.log(f"Researcher: arXiv search error (may be rate-limited): {e}")

    # Basic summarization (simple concatenation for scaffold)
    combined = "\n\n".join([r["snippet"] for r in results])[:3000]
    task.results = results
    task.summary = combined or f"No results found for {task.query}"
    # charge token budget estimate
    state.token_budget.charge(estimate_tokens(task.summary))
    task.status = "done"
    state.log(f"Researcher: finished task '{task.title}', results={len(results)}. Tokens used: {state.token_budget.used}")
    # add to bibliography (simple)
    for r in results:
        state.bibliography.append({"title": r.get("title"), "source": r.get("source"), "url": r.get("url")})
    return task


In [8]:
# Cell 7: Writer Agent
def writer_agent(state: ProjectState) -> Tuple[str, ProjectState]:
    """
    Create a cohesive Markdown draft from completed tasks in state.plan.
    For production: call an LLM to synthesize with citations. Here we build a template.
    """
    state.log("Writer: synthesizing draft from research.")
    md_parts = [f"# Literature Review: {state.prompt}\n\n"]
    for t in state.plan:
        md_parts.append(f"## {t.section}\n")
        md_parts.append(f"{t.summary or 'No summary available.'}\n\n")
        # inline citations (simple numbered)
        for i, r in enumerate(t.results, start=1):
            md_parts.append(f"[{i}] {r.get('title','')} — {r.get('source')} — {r.get('url')}\n")
        md_parts.append("\n---\n")
    draft = "\n".join(md_parts)
    # token charge
    state.token_budget.charge(estimate_tokens(draft))
    state.final_draft = draft
    state.log("Writer: draft created.")
    return draft, state


In [9]:
# Cell 8: Reviewer Agent
def reviewer_agent(draft: str, state: ProjectState) -> Tuple[bool, str]:
    """
    Automated reviewer: checks for
     - presence of bibliography
     - any obviously empty sections
     - basic hallucination check (Do cited URLs exist in bibliography)
    Returns (passes_review:bool, feedback:str)
    """
    state.log("Reviewer: starting automated quality check.")
    feedback = []
    if not state.bibliography:
        feedback.append("No bibliography entries detected.")
    # check sections length
    if len(draft) < 500:
        feedback.append("Draft seems short (<500 chars).")
    # simplistic hallucination check: ensure any 'url' substrings exist
    if "http" in draft and not state.bibliography:
        feedback.append("Found inline URLs but bibliography is empty.")
    passes = len(feedback) == 0
    state.log(f"Reviewer: passes={passes}; feedback={feedback}")
    return passes, ("\n".join(feedback) if feedback else "All checks passed.")


In [10]:
# Cell 9: HIL Node 2 (Final Draft acceptance)
def hil2_review(draft: str) -> Tuple[bool, Optional[str]]:
    """
    Present final draft to human. Accept or reject with critique.
    """
    print("\n=== DRAFT PREVIEW (first 800 chars) ===\n")
    print(draft[:800])
    print("\n=== END PREVIEW ===\n")
    resp = input("HIL2: Accept final draft? (y/n): ").strip().lower()
    if resp == 'y':
        return True, None
    else:
        fb = input("Please provide critique/required changes (short): ").strip()
        return False, fb


In [11]:
# Cell 10: Orchestrator (local fallback or LangGraph if available)

def run_pipeline(project_prompt: str, state: Optional[ProjectState]=None):
    if state is None:
        state = ProjectState(project_id=str(uuid.uuid4()), prompt=project_prompt)
    # 1. Planner
    state, plan_meta = planner_agent(state)
    accepted, feedback = hil1_review(plan_meta)
    if not accepted:
        state.log("HIL1 rejected plan; adding feedback and re-planning.")
        state.log(f"HIL1 feedback: {feedback}")
        # For simplicity we update prompt and re-run planner once (could route feedback into planner prompt)
        state.prompt += " | feedback: " + feedback
        state, plan_meta = planner_agent(state)
        accepted, feedback = hil1_review(plan_meta)
        if not accepted:
            state.log("HIL1 rejected plan twice. Aborting.")
            return state

    # 2. Researcher: execute tasks sequentially
    for t in state.plan:
        if t.status != "done":
            researcher_agent(t, state)

    # 3. Writer
    draft, state = writer_agent(state)

    # 4. Reviewer (automated)
    passes, review_feedback = reviewer_agent(draft, state)
    if not passes:
        state.log("Reviewer failed automated checks; sending back to Writer for revision.")
        # simple auto-fix: append review feedback to each task summary and rewrite
        for t in state.plan:
            t.summary = (t.summary or "") + "\n\n/* Reviewer notes: " + review_feedback + " */"
        draft, state = writer_agent(state)
        passes, review_feedback = reviewer_agent(draft, state)
        if not passes:
            state.log("Reviewer still failing after one auto-rewrite; escalating to HIL2 for instructions.")

    # 5. HIL2 final acceptance
    accepted, final_feedback = hil2_review(draft)
    state.iterations += 1

    while (not accepted) and state.iterations < state.max_iterations:
        state.log(f"HIL2 rejected draft. Feedback: {final_feedback}")
        # Route feedback: send back to Planner with feedback to produce a targeted re-plan
        state.prompt += " | final_feedback: " + final_feedback
        state, plan_meta = planner_agent(state)
        state.log("Re-running research & writing cycle after HIL2 feedback.")
        # Re-run researcher for tasks (could smartly only re-run affected tasks; here we run all)
        for t in state.plan:
            researcher_agent(t, state)
        draft, state = writer_agent(state)
        passes, review_feedback = reviewer_agent(draft, state)
        if not passes:
            state.log("Reviewer failed after revision; adding notes into draft.")
            draft += "\n\n<!-- Reviewer notes: " + review_feedback + " -->"
        accepted, final_feedback = hil2_review(draft)
        state.iterations += 1

    if accepted:
        state.log("HIL2 accepted final draft. Pipeline complete.")
    else:
        state.log("Iteration limit reached or HIL2 not accepting. Pipeline stopped.")
    return state

# Example run (interactive)
# project_prompt = "Explain deep learning methods for plant disease detection (focus on 2020-2025)"
# final_state = run_pipeline(project_prompt)


In [12]:
# Cell 11: Generate architecture diagram
def render_architecture_diagram(output_file="architecture.gv"):
    dot = Digraph(comment='HIL Multi-agent Pipeline', format='png')
    dot.node('User', 'User (initial prompt)')
    dot.node('Planner', 'Planner Agent')
    dot.node('HIL1', 'Human Admin (Plan Approval)')
    dot.node('Researcher', 'Researcher Agent')
    dot.node('Writer', 'Writer Agent')
    dot.node('Reviewer', 'Reviewer Agent')
    dot.node('HIL2', 'Human Admin (Final Acceptance)')
    dot.node('State', 'State Schema & Token Budget')

    dot.edges([('User','Planner'), ('Planner','HIL1'), ('HIL1','Researcher'), ('Researcher','Writer'),
               ('Writer','Reviewer'), ('Reviewer','HIL2')])
    dot.edge('State','Planner', label='context')
    dot.edge('State','Researcher', label='task list')
    dot.edge('HIL2','Planner', label='reject feedback (loop)', style='dashed')
    dot.edge('Reviewer','Writer', label='fail -> rewrite', style='dashed')
    dot.render(output_file, view=False)
    print(f"Diagram rendered to {output_file}.png")

render_architecture_diagram("hil_architecture")


Diagram rendered to hil_architecture.png


In [17]:
# Cell 12: Improved Non-interactive demo run (with debug, timing, loop protection)

import time
import builtins

# --- CONFIG ---
MAX_PIPELINE_ITER = 5   # prevents infinite loops
MAX_REVIEWER_REVISION = 2  # prevents Writer <-> Reviewer looping

# Helper functions to adapt agent calls for safe_run_pipeline structure
# These mirror the logic from the main run_pipeline but are tailored for the
# non-interactive, loop-controlled safe_run_pipeline within this cell.
def run_planner_wrapper(prompt_str: str, state: ProjectState) -> ProjectState:
    """Wrapper for planner_agent, assuming HIL1 is auto-accepted via monkey-patch."""
    # The actual planning uses state.prompt. prompt_str is passed for consistent signature.
    state.log("Planner wrapper: calling planner_agent.")
    state, plan_meta = planner_agent(state)
    # HIL1 review is handled by the monkey-patch of input() in demo_run.
    # If a real HIL1 interaction were needed, it would be here.
    return state

def run_researcher_wrapper(prompt_str: str, state: ProjectState) -> ProjectState:
    """Wrapper for researcher_agent, running all pending tasks."""
    state.log("Researcher wrapper: executing tasks.")
    for t in state.plan:
        if t.status != "done":
            researcher_agent(t, state)
    return state

def run_writer_wrapper(prompt_str: str, state: ProjectState) -> ProjectState:
    """Wrapper for writer_agent, updating final_draft in state."""
    state.log("Writer wrapper: synthesizing draft.")
    draft, updated_state = writer_agent(state)
    updated_state.final_draft = draft # Ensure final_draft is correctly set in the state
    return updated_state

def run_reviewer_wrapper(prompt_str: str, state: ProjectState) -> ProjectState:
    """Wrapper for reviewer_agent, setting needs_revision flag and applying feedback."""
    state.log("Reviewer wrapper: performing automated review.")
    passes, review_feedback = reviewer_agent(state.final_draft, state)
    if not passes:
        state.log(f"Reviewer failed automated checks. Feedback: {review_feedback}")
        setattr(state, "needs_revision", True)
        setattr(state, "review_feedback", review_feedback)
        # Apply simple auto-fix: append review feedback to each task summary for next rewrite
        # This mimics the behavior of the full `run_pipeline` on reviewer rejection.
        for t in state.plan:
            t.summary = (t.summary or "") + "\n\n/* Reviewer notes: " + review_feedback + " */"
    else:
        state.log("Reviewer accepted output.")
        setattr(state, "needs_revision", False)
        setattr(state, "review_feedback", "")
    return state


def demo_run():
    print("🔄 Starting demo pipeline...\n")

    ps = ProjectState(
        project_id="demo-001",
        prompt="Plant disease detection using CNNs and transfer learning (2020-2024)"
    )

    # Auto-accept HIL steps
    real_input = builtins.input
    # Provide enough 'y' for all potential input calls in loops.
    # The safe_run_pipeline itself is non-interactive but underlying agents might be configured to call input.
    answers = iter(["y", "y", "y", "y", "y", "y"])
    builtins.input = lambda prompt="": next(answers)

    try:
        final_state = safe_run_pipeline(ps.prompt, ps)
    except StopIteration:
        # This can happen if the 'answers' iterator runs out of 'y's.
        # It means some HIL step required more input than provided.
        print("Error: Not enough 'y' inputs provided for HIL steps. Increase 'answers' iterator size.")
        final_state = ps
    finally:
        builtins.input = real_input

    print("\n✅ Demo run finished.")
    return final_state


# -------- SAFE PIPELINE WRAPPER ----------
def safe_run_pipeline(prompt, state):
    print("⏳ Pipeline started ...")
    start_time = time.time()

    reviewer_rejects = 0

    for iteration in range(1, MAX_PIPELINE_ITER + 1):
        print(f"\n========== ITERATION {iteration} ==========")

        iter_start = time.time()

        # --- Planner ---
        print("🧠 Running Planner...")
        t0 = time.time()
        state = run_planner_wrapper(prompt, state) # Use wrapper
        print(f"   ✔ Planner done in {time.time() - t0:.2f}s")

        # --- Research ---
        print("🔍 Running Researcher...")
        t0 = time.time()
        state = run_researcher_wrapper(prompt, state) # Use wrapper
        print(f"   ✔ Researcher done in {time.time() - t0:.2f}s")

        # --- Writer ---
        print("✍️  Running Writer...")
        t0 = time.time()
        state = run_writer_wrapper(prompt, state) # Use wrapper
        print(f"   ✔ Writer done in {time.time() - t0:.2f}s")

        # --- Reviewer ---
        print("📝 Running Reviewer...")
        t0 = time.time()
        state = run_reviewer_wrapper(prompt, state) # Use wrapper
        print(f"   ✔ Reviewer done in {time.time() - t0:.2f}s")

        # Check if Reviewer asked for revision
        if getattr(state, "needs_revision", False):
            reviewer_rejects += 1
            print(f"⚠ Reviewer rejected — revision count: {reviewer_rejects}")

            if reviewer_rejects >= MAX_REVIEWER_REVISION:
                print("🚫 Too many reviewer rejections — forcing acceptance for pipeline completion.")
                state.needs_revision = False
        else:
            print("🎉 Reviewer accepted output. Pipeline complete.")
            break

        print(f"Iteration time: {time.time() - iter_start:.2f}s")

    print(f"\n⏱ Total pipeline time: {time.time() - start_time:.2f}s")
    return state


# ---------------- RUN DEMO -----------------

demo_state = demo_run()
print("Demo run complete. Iterations:", demo_state.iterations)
print("Final draft snippet:\n", demo_state.final_draft[:800])

🔄 Starting demo pipeline...

⏳ Pipeline started ...

========== ITERATION 1 ==========
🧠 Running Planner...
   ✔ Planner done in 0.00s
🔍 Running Researcher...


/tmp/ipython-input-2703690830.py:34: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  for r in search.results():


   ✔ Researcher done in 274.91s
✍️  Running Writer...
   ✔ Writer done in 0.00s
📝 Running Reviewer...
   ✔ Reviewer done in 0.00s
🎉 Reviewer accepted output. Pipeline complete.

⏱ Total pipeline time: 274.91s

✅ Demo run finished.
Demo run complete. Iterations: 0
Final draft snippet:
 # Literature Review: Plant disease detection using CNNs and transfer learning (2020-2024)


## History / background

This is a list of datasets for machine learning research. It is part of the list of datasets for machine-learning research. These datasets consist primarily of images or videos for tasks such as object detection, facial recognition, and multi-label classification.

A number of significant scientific events occurred in 2020.


== Events ==


=== January ===


=== February ===


=== March ===


=== April ===


=== May ===


=== June ===


=== July ===


=== August ===


=== September ===


=== October ===

1 October
Researchers report the discovery of a novel overlapping gene (OLG) (a gene pa

In [18]:
# Cell 13: Save outputs
def save_outputs(state: ProjectState, folder="/content/hil_output"):
    os.makedirs(folder, exist_ok=True)
    md_path = os.path.join(folder, f"{state.project_id}_draft.md")
    bib_path = os.path.join(folder, f"{state.project_id}_bibliography.json")
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(state.final_draft or "")
        f.write("\n\n")
        f.write("## Bibliography\n")
        for i, b in enumerate(state.bibliography, start=1):
            f.write(f"{i}. {b.get('title')} — {b.get('source')} — {b.get('url')}\n")
    with open(bib_path, "w", encoding="utf-8") as f:
        json.dump(state.bibliography, f, indent=2)
    print("Saved:", md_path, bib_path)

save_outputs(demo_state)


Saved: /content/hil_output/demo-001_draft.md /content/hil_output/demo-001_bibliography.json


In [19]:
# Colab-ready Gradio UI for HIL Multi-Agent Pipeline
# Save this file into your Colab environment (or paste into a Colab cell) and run.
# It depends on the notebook scaffold where ProjectState, planner_agent, researcher_agent,
# writer_agent, reviewer_agent, save_outputs, render_architecture_diagram, demo_run, etc. are defined.

import os
import time
import json
import threading
from typing import Tuple

# Gradio UI
try:
    import gradio as gr
except Exception:
    raise ImportError("Please pip install gradio in Colab: !pip install gradio")

# Ensure /content/hil_output exists
OUT_DIR = "/content/hil_output"
os.makedirs(OUT_DIR, exist_ok=True)
ARCH_PNG = "/content/hil_architecture.png"

# We'll use the functions from the notebook environment. If you pasted this into the same Colab runtime,
# the functions and classes (ProjectState, planner_agent, researcher_agent, writer_agent, reviewer_agent,
# run_pipeline, save_outputs, demo_run) should already be available. If not, import or paste those cells first.

# global runtime state
_runtime = {
    'state': None,
    'logs': []
}

# Helpers to append to logs and keep UI responsive
def append_log(msg: str):
    t = time.strftime("%Y-%m-%d %H:%M:%S")
    _runtime['logs'].append(f"[{t}] {msg}")

# UI Actions

def generate_plan(prompt: str) -> Tuple[str, dict, str]:
    """Run planner_agent and return plan text, plan metadata, and logs."""
    append_log("Planner started")
    try:
        state = ProjectState(project_id=str(time.time()), prompt=prompt)
        state, plan_meta = planner_agent(state)
        _runtime['state'] = state
        plan_text = "\n".join([f"- {p['title']}: {p['query']}" for p in plan_meta['plan']])
        append_log("Planner finished")
        return plan_text, plan_meta, '\n'.join(state.logs[-10:])
    except Exception as e:
        append_log(f"Planner error: {e}")
        return "", {}, '\n'.join(state.logs[-10:])


def proceed_with_research(plan_meta: dict, hil1_accept: bool, hil1_feedback: str) -> Tuple[str,str,str]:
    """If HIL1 accepted, run researcher -> writer -> reviewer (synchronously) and return draft, bibliography, logs."""
    if not hil1_accept:
        append_log("HIL1 rejected plan. Feedback added to prompt and replanning.")
        # incorporate feedback and re-run planner
        state = _runtime.get('state')
        if state is None:
            return "No state found","", ""
        state.prompt += " | feedback: " + (hil1_feedback or "no feedback")
        state, plan_meta = planner_agent(state)
        _runtime['state'] = state
    state = _runtime.get('state')
    append_log("Researcher started")
    # run researcher for each task
    for t in state.plan:
        if t.status != 'done':
            researcher_agent(t, state)
    append_log("Researcher finished")
    append_log("Writer started")
    draft, state = writer_agent(state)
    append_log("Writer finished")
    append_log("Reviewer started")
    passes, fb = reviewer_agent(draft, state)
    state.needs_revision = not passes
    append_log(f"Reviewer finished: passes={passes}")
    _runtime['state'] = state
    bib_text = '\n'.join([f"- {b.get('title')} ({b.get('source')}) - {b.get('url')}" for b in state.bibliography])
    return draft, bib_text, '\n'.join(state.logs[-20:])


def human_final_review(hil2_accept: bool, hil2_feedback: str) -> Tuple[str,str]:
    """Handle HIL2 decisions: either finish or loop back to planner with feedback."""
    state = _runtime.get('state')
    if state is None:
        return "", "No project state found."
    if hil2_accept:
        append_log("HIL2 accepted final draft.")
        # save outputs
        save_outputs(state, folder=OUT_DIR)
        return "Accepted and saved.", '\n'.join(state.logs[-20:])
    else:
        append_log("HIL2 rejected final draft. Feedback added to prompt.")
        state.prompt += " | final_feedback: " + (hil2_feedback or "no feedback")
        _runtime['state'] = state
        return "Rejected; updated plan. Click 'Generate Plan' to re-plan.", '\n'.join(state.logs[-20:])


def start_demo() -> Tuple[str,str,str,str]:
    """Run demo_run (non-interactive), then expose files and logs."""
    append_log("Starting demo run (auto-accept HILs)")
    demo_state = demo_run()
    _runtime['state'] = demo_state
    # save
    save_outputs(demo_state, folder=OUT_DIR)
    md_file = os.path.join(OUT_DIR, f"{demo_state.project_id}_draft.md")
    bib_file = os.path.join(OUT_DIR, f"{demo_state.project_id}_bibliography.json")
    files_list = '\n'.join(os.listdir(OUT_DIR))
    append_log("Demo run complete and files saved.")
    preview = (demo_state.final_draft or '')[:1500]
    return preview, files_list, bib_file, '\n'.join(demo_state.logs[-30:])


def list_generated_files() -> str:
    items = os.listdir(OUT_DIR)
    return '\n'.join(items)


# Build Gradio interface
with gr.Blocks(title="HIL Multi-Agent Research System", css=".output{height:300px}") as demo:
    gr.Markdown("# Self-Correcting Autonomous Research System (HIL) — UI")
    with gr.Row():
        with gr.Column(scale=2):
            prompt_in = gr.Textbox(label="Project prompt", placeholder="e.g. Plant disease detection using CNNs (2020-2024)", lines=2)
            gen_plan_btn = gr.Button("Generate Plan")
            plan_md = gr.Textbox(label="Plan (preview)", lines=6)
            hil1_radio = gr.Radio(["accept","reject"], label="HIL1: Plan decision", value="accept")
            hil1_feedback = gr.Textbox(label="HIL1 feedback (if reject)", lines=2)
            run_research_btn = gr.Button("Run Research & Draft")
            draft_output = gr.Textbox(label="Draft (preview)", lines=20)
            bib_output = gr.Textbox(label="Bibliography", lines=6)
            hil2_radio = gr.Radio(["accept","reject"], label="HIL2: Final decision", value="accept")
            hil2_feedback = gr.Textbox(label="HIL2 feedback (if reject)", lines=2)
            final_btn = gr.Button("Finalize & Save")
            save_status = gr.Textbox(label="Save status")
        with gr.Column(scale=1):
            # Architecture image
            if os.path.exists(ARCH_PNG):
                arch_img = gr.Image(value=ARCH_PNG, label="Architecture Diagram")
            else:
                arch_img = gr.Markdown("Architecture diagram not found. Run render_architecture_diagram() in the notebook to generate hil_architecture.png")
            gr.Markdown("### Generated files (in /content/hil_output)")
            files_box = gr.Textbox(value=list_generated_files(), label="Files")
            refresh_files = gr.Button("Refresh file list")
            gr.Markdown("### Execution Logs")
            logs_box = gr.Textbox(value="", label="Logs (latest)", lines=20)
            demo_btn = gr.Button("Run Demo (auto-accept HILs)")
            demo_preview = gr.Textbox(label="Demo draft preview", lines=12)
            demo_files = gr.Textbox(label="Demo generated files")

    # Wire callbacks
    def on_gen_plan(prompt):
        plan_text, plan_meta, logs = generate_plan(prompt)
        return plan_text, logs
    gen_plan_btn.click(on_gen_plan, inputs=[prompt_in], outputs=[plan_md, logs_box])

    def on_run_research(plan_md_text, hil1_choice, hil1_fb):
        # parse plan_meta from runtime state
        state = _runtime.get('state')
        plan_meta = {"plan": [{"title": t.title, "query": t.query, "id": t.id, "section": t.section} for t in state.plan]} if state else {"plan": []}
        draft, bib, logs = proceed_with_research(plan_meta, hil1_choice == 'accept', hil1_fb)
        # update files list
        return draft[:2000], bib, logs
    run_research_btn.click(on_run_research, inputs=[plan_md, hil1_radio, hil1_feedback], outputs=[draft_output, bib_output, logs_box])

    def on_finalize(hil2_choice, hil2_fb):
        status, logs = human_final_review(hil2_choice == 'accept', hil2_fb)
        return status, logs
    final_btn.click(on_finalize, inputs=[hil2_radio, hil2_feedback], outputs=[save_status, logs_box])

    def on_refresh_files():
        return list_generated_files()
    refresh_files.click(on_refresh_files, outputs=[files_box])

    demo_btn.click(start_demo, outputs=[demo_preview, demo_files, demo_files, logs_box])

# Launch the interface
if __name__ == "__main__":
    demo.launch(share=False)


/tmp/ipython-input-196833562.py:123: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="HIL Multi-Agent Research System", css=".output{height:300px}") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [21]:
# ================================================================
# ChatGPT-style Interactive UI for Multi-Agent HIL Research System
# ================================================================

import gradio as gr
import json
import os
import time
import base64

# Utility to read files safely
def load_file(path):
    if not os.path.exists(path):
        return "File not found."
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

# Utility: typing animation
def typing_effect(text, delay=0.02):
    output = ""
    for char in text:
        output += char
        time.sleep(delay)
        yield output


# Global iteration counter to mock “subscription alert”
MAX_ITER = 3


# ================================================================
# Main Chat function (behaves like ChatGPT)
# ================================================================
def chat_interface(user_message, history):

    # Initialize history if needed
    if history is None:
        history = []

    # Check if project state exists
    global ps
    if ps is None:
        history.append(("system", "⚠️ Please click **Start New Project** first."))
        return history

    # -------------------- Planner Phase --------------------
    if "plan" not in ps.__dict__ or ps.plan is None:
        history.append(("user", user_message))
        history.append(("assistant", "🧠 Generating research plan..."))

        for t in typing_effect("Thinking..."):
            pass

        plan_state = planner_agent(ps.prompt)
        ps.update_plan(plan_state.plan)

        history.append(("assistant",
            f"📌 **Proposed Research Plan:**\n\n```json\n{json.dumps(ps.plan, indent=2)}\n```\n"
            f"Would you like to **approve** or **reject** this plan?"
        ))
        return history

    # -------------------- HIL 1: Plan approval --------------------
    if user_message.lower() in ["approve plan", "approve", "y"]:
        history.append(("assistant", "✅ Plan approved.\n\n🔍 Starting Research..."))

        res_state = researcher_agent(ps.plan)
        ps.update_research(res_state.research_results)

        history.append(("assistant", "📚 Research complete.\n\nType **continue** to generate draft."))
        return history

    if user_message.lower() in ["reject plan", "reject", "no"]:
        history.append(("assistant", "❌ Plan rejected. Please provide feedback:"))
        ps.plan = None
        return history

    # -------------------- Writer Phase --------------------
    if user_message.lower() in ["continue", "write", "generate draft"]:
        history.append(("assistant", "✍️ Writing draft... please wait."))

        draft_state = writer_agent(ps.research_results)
        ps.update_draft(draft_state.draft)

        history.append(("assistant",
            f"📝 **Draft Generated:**\n\n```markdown\n{ps.latest_draft[:1200]}...\n```"
            f"\nShould the system **review** the draft?"
        ))
        return history

    # -------------------- Reviewer Phase --------------------
    if user_message.lower() in ["review", "review draft"]:
        ps.iterations += 1

        # Show subscription popup if too many iterations
        if ps.iterations > MAX_ITER:
            history.append(("assistant",
                "⚠️ **You have reached the free iteration limit.**\n"
                "Upgrade to **Pro Research Plan** to continue 🔓\n\n"
                "👉 (This is a mock popup requested by you.)"
            ))
            return history

        history.append(("assistant", "🔎 Reviewer evaluating draft..."))

        rev_state = reviewer_agent(ps.latest_draft)

        if rev_state.needs_revision:
            history.append(("assistant",
                f"⚠️ Reviewer Feedback:\n\n{rev_state.feedback}\n\n"
                "Would you like to **revise** the draft?"
            ))
            return history

        history.append(("assistant", "🎉 Draft approved by Reviewer!"))
        history.append(("assistant", "Would you like to **Final Accept** or **Final Reject**?"))
        return history

    # -------------------- HIL 2: Final Accept / Reject --------------------
    if user_message.lower() == "final accept":

        # Save output files
        os.makedirs("/content/hil_output", exist_ok=True)
        md_path = "/content/hil_output/final.md"
        biblio_path = "/content/hil_output/bibliography.json"
        graph_path = "/content/hil_output/architecture_graph.png"

        with open(md_path, "w") as f:
            f.write(ps.latest_draft)

        with open(biblio_path, "w") as f:
            json.dump(ps.bibliography, f, indent=2)

        # If graph exists
        graph_exists = os.path.exists(graph_path)

        links = []
        links.append(f"📄 [Download Markdown Output](file={md_path})")
        links.append(f"📚 [Download Bibliography](file={biblio_path})")
        if graph_exists:
            links.append(f"🧩 [Download Architecture Graph](file={graph_path})")

        history.append(("assistant",
            "🎉 **Final Output Ready!**\n\n" +
            "\n".join(links)
        ))
        return history

    if user_message.lower() == "final reject":
        history.append(("assistant",
            "❌ Final Draft Rejected.\nPlease provide feedback to generate a new plan."
        ))
        ps.plan = None
        return history

    # ----------------------- Fallback -----------------------
    history.append(("assistant", "🤖 I didn’t understand that. Try: approve plan / review / continue"))
    return history


# ===========================================================
# UI Layout
# ===========================================================
with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue")) as demo:

    gr.Markdown("<h1 style='text-align:center'>🔬 Multi-Agent ChatGPT-Style Research Assistant</h1>")
    gr.Markdown("Interactively generate **research articles**, with HIL approval gates.")

    with gr.Row():
        with gr.Column():
            topic_box = gr.Textbox(label="Research Topic", placeholder="e.g., Plant disease detection using CNNs (2020–2024)")
            start_btn = gr.Button("Start New Project 🧠")
        with gr.Column():
            clear_btn = gr.Button("Clear Chat")

    chatbot = gr.Chatbot(height=600)

    user_input = gr.Textbox(label="Message", placeholder="Type your message here...")

    # State holder
    state_holder = gr.State([])

    # Start new project
    def start_project(topic):
        global ps
        ps = ProjectState(project_id="chat-" + str(int(time.time())), prompt=topic)
        return [("assistant", "Project initialized. Type **hi** to begin planning.")]

    start_btn.click(start_project, inputs=topic_box, outputs=chatbot)

    clear_btn.click(lambda: [], outputs=chatbot)

    user_input.submit(chat_interface, [user_input, chatbot], chatbot)


demo.launch(debug=True)


/tmp/ipython-input-1654847245.py:164: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue")) as demo:
/tmp/ipython-input-1654847245.py:176: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=600)
/tmp/ipython-input-1654847245.py:176: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=600)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b989ef42a7c507d4dc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://08730c28bdef1a63eb.gradio.live
Killing tunnel 127.0.0.1:7862 <> https://b989ef42a7c507d4dc.gradio.live
